# Brahmacharini — from observed behaviour to the ATT&CK technique

Model 2 of the nine. **Data:** 17.6k MITRE ATT&CK procedure examples, split **by source** (attackers never seen in training). **Method:** a dual encoder written from scratch — hashed uni/bigrams → `EmbeddingBag` → 128-d — trained with InfoNCE. **Baselines:** BM25 and Smriti's `HashingEmbedder`.

Results: `docs/research/models/brahmacharini.md`; decision: `docs/adr/SatiEmbed.md`.

In [ ]:
# Run on Kaggle or Colab. Clones this repository and installs the two packages.
import os, subprocess, sys
if not os.path.exists("PASHUPATASTRA"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/preetraval45/PASHUPATASTRA.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "PASHUPATASTRA/packages/core", "-e", "PASHUPATASTRA/packages/sati", "matplotlib"], check=True)
sys.path[:0] = ["PASHUPATASTRA/packages/core", "PASHUPATASTRA/packages/sati"]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'], check=True)

In [ ]:
from pathlib import Path
from sati.embed import data as d
techniques, procedures = d.load(Path("cache/corpus"))
train, val, test = d.split_by_source(procedures)
print(len(techniques), "techniques;", len(train), len(val), len(test), "procedures")
test[0]

In [ ]:
import subprocess, sys, json, pandas as pd
subprocess.run([sys.executable, "-m", "sati.embed.run", "--cache", "cache/corpus", "--out", "results/brahmacharini"], check=True, cwd="PASHUPATASTRA/packages/sati")
r = json.load(open("PASHUPATASTRA/packages/sati/results/brahmacharini/results.json"))
pd.DataFrame(r["test"]).T

## Try it — ask which technique a behaviour is

In [ ]:
import torch
from sati.embed.model import Encoder
model = Encoder(); model.load_state_dict(torch.load("PASHUPATASTRA/packages/sati/results/brahmacharini/brahmacharini-v0.pt")); model.eval()
query = "the malware encrypted files on network shares and left a ransom note"
with torch.no_grad():
    scores = (model([query]) @ model([t.text for t in techniques]).T)[0]
for i in scores.argsort(descending=True)[:5]:
    print(f"{scores[i]:.3f}", techniques[i].attack_id, techniques[i].text[:80])